# Demo 1: Exploring Heart Rate Patterns During Meditation

In this demo, we'll explore real heart rate data from the Heart Rate Oscillations during Meditation dataset on PhysioNet. We'll learn how to:
- Load and visualize real physiological time series data
- Identify patterns in heart rate during different meditation techniques
- Perform basic statistical tests on time series data
- Handle missing values and irregularities in real-world data

## 1. Setting Up and Loading the Data

First, let's import the necessary libraries and download the dataset from PhysioNet.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from urllib.request import urlretrieve

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Create a directory for data if it doesn't exist
import os
os.makedirs('data', exist_ok=True)

In [ ]:
# Download data from PhysioNet
base_url = "https://physionet.org/files/meditation/1.0.0/"

# Download metadata file
metadata_url = base_url + "metadata.csv"
metadata_file = "data/meditation_metadata.csv"
urlretrieve(metadata_url, metadata_file)

# Download data for a few subjects
subjects = ['s001', 's002', 's003']
data_files = {}

for subject in subjects:
    data_url = base_url + f"data/{subject}.txt"
    local_file = f"data/{subject}.txt"
    urlretrieve(data_url, local_file)
    data_files[subject] = local_file

print(f"Downloaded data for {len(subjects)} subjects")

Now let's load the metadata and understand what's in the dataset.

In [ ]:
# Load metadata
metadata = pd.read_csv(metadata_file)

# Display the first few rows
print("Dataset metadata:")
metadata.head()

In [ ]:
# Let's see what meditation techniques are included
print("\nMeditation techniques in the dataset:")
print(metadata['meditation_technique'].value_counts())

Now let's load the heart rate data for one subject and examine it.

In [ ]:
# Load data for the first subject
subject_id = subjects[0]
subject_file = data_files[subject_id]

# Load the data (tab-separated values with no header)
# Column 1: Time (seconds)
# Column 2: Heart rate (beats per minute)
hr_data = pd.read_csv(subject_file, sep='\t', header=None, names=['time', 'heart_rate'])

# Convert time to datetime
start_time = pd.Timestamp('2000-01-01')  # Arbitrary start time
hr_data['datetime'] = start_time + pd.to_timedelta(hr_data['time'], unit='s')

# Display the first few rows
print(f"Heart rate data for subject {subject_id}:")
hr_data.head()

In [ ]:
# Basic statistics
print(f"Data points: {len(hr_data)}")
print(f"Time range: {hr_data['time'].min():.1f} to {hr_data['time'].max():.1f} seconds")
print(f"Heart rate range: {hr_data['heart_rate'].min():.1f} to {hr_data['heart_rate'].max():.1f} bpm")
print(f"Mean heart rate: {hr_data['heart_rate'].mean():.1f} bpm")
print(f"Standard deviation: {hr_data['heart_rate'].std():.1f} bpm")

## 2. Visualizing Heart Rate Patterns

Let's visualize the heart rate data to identify patterns during meditation.

In [ ]:
# Plot the heart rate time series
plt.figure(figsize=(12, 6))
plt.plot(hr_data['datetime'], hr_data['heart_rate'], 'b-', alpha=0.7)
plt.title(f'Heart Rate During Meditation - Subject {subject_id}')
plt.xlabel('Time')
plt.ylabel('Heart Rate (bpm)')
plt.grid(True)
plt.show()

Let's smooth the data to see the underlying patterns better.

In [ ]:
# Calculate rolling average (30-second window)
window_size = 30  # Adjust based on sampling rate
hr_data['hr_rolling'] = hr_data['heart_rate'].rolling(window=window_size).mean()

# Plot raw and smoothed data
plt.figure(figsize=(12, 6))
plt.plot(hr_data['datetime'], hr_data['heart_rate'], 'b-', alpha=0.3, label='Raw')
plt.plot(hr_data['datetime'], hr_data['hr_rolling'], 'r-', linewidth=2, label='30-second Rolling Average')
plt.title(f'Heart Rate During Meditation - Subject {subject_id}')
plt.xlabel('Time')
plt.ylabel('Heart Rate (bpm)')
plt.legend()
plt.grid(True)
plt.show()

Now let's compare heart rate patterns across different subjects.

In [ ]:
# Load data for all subjects
all_subjects_data = {}

for subject in subjects:
    # Load data
    subject_file = data_files[subject]
    data = pd.read_csv(subject_file, sep='\t', header=None, names=['time', 'heart_rate'])
    
    # Add subject info
    data['subject'] = subject
    
    # Get meditation technique
    technique = metadata[metadata['subject_id'] == subject]['meditation_technique'].values[0]
    data['technique'] = technique
    
    # Store in dictionary
    all_subjects_data[subject] = data

# Combine all data
combined_data = pd.concat(all_subjects_data.values())

# Plot heart rate distributions by subject
plt.figure(figsize=(10, 6))
sns.boxplot(x='subject', y='heart_rate', data=combined_data)
plt.title('Heart Rate Distribution by Subject')
plt.xlabel('Subject ID')
plt.ylabel('Heart Rate (bpm)')
plt.grid(True)
plt.show()

In [ ]:
# Plot heart rate distributions by meditation technique
plt.figure(figsize=(10, 6))
sns.boxplot(x='technique', y='heart_rate', data=combined_data)
plt.title('Heart Rate Distribution by Meditation Technique')
plt.xlabel('Meditation Technique')
plt.ylabel('Heart Rate (bpm)')
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Statistical Analysis of Heart Rate Time Series

Let's perform some statistical tests to analyze the heart rate time series.

In [ ]:
# Test for stationarity using Augmented Dickey-Fuller test
def test_stationarity(timeseries):
    # Calculate rolling statistics
    rolling_mean = timeseries.rolling(window=30).mean()
    rolling_std = timeseries.rolling(window=30).std()
    
    # Plot rolling statistics
    plt.figure(figsize=(12, 6))
    plt.plot(timeseries, label='Original')
    plt.plot(rolling_mean, label='Rolling Mean')
    plt.plot(rolling_std, label='Rolling Std')
    plt.legend()
    plt.title('Rolling Mean & Standard Deviation')
    plt.grid(True)
    plt.show()
    
    # Perform Dickey-Fuller test
    print('Results of Dickey-Fuller Test:')
    dftest = adfuller(timeseries.dropna())
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])
    for key, value in dftest[4].items():
        dfoutput['Critical Value (%s)'%key] = value
    print(dfoutput)
    
    # Interpret the results
    if dftest[1] <= 0.05:
        print("\nConclusion: The time series is stationary (p-value <= 0.05)")
    else:
        print("\nConclusion: The time series is not stationary (p-value > 0.05)")

# Test stationarity for the first subject
test_stationarity(hr_data['heart_rate'])

In [ ]:
# Plot autocorrelation and partial autocorrelation
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Autocorrelation
plot_acf(hr_data['heart_rate'].dropna(), lags=50, ax=ax1)
ax1.set_title('Autocorrelation Function')

# Partial Autocorrelation
plot_pacf(hr_data['heart_rate'].dropna(), lags=50, ax=ax2)
ax2.set_title('Partial Autocorrelation Function')

plt.tight_layout()
plt.show()

## 4. Handling Missing Values and Irregularities

Let's introduce some artificial missing values to demonstrate how to handle them.

In [ ]:
# Create a copy of the data with artificial missing values
hr_missing = hr_data.copy()
np.random.seed(42)

# Randomly set 10% of values to NaN
mask = np.random.random(len(hr_missing)) < 0.1
hr_missing.loc[mask, 'heart_rate'] = np.nan

# Display the first few rows with missing values
print("Data with artificial missing values:")
hr_missing.head(10)

In [ ]:
# Plot data with missing values
plt.figure(figsize=(12, 6))
plt.plot(hr_missing['datetime'], hr_missing['heart_rate'], 'b-', alpha=0.7)
plt.title('Heart Rate with Missing Values')
plt.xlabel('Time')
plt.ylabel('Heart Rate (bpm)')
plt.grid(True)
plt.show()

Now let's try different imputation methods to handle the missing values.

In [ ]:
# Different imputation strategies
hr_imputed = pd.DataFrame({
    'datetime': hr_missing['datetime'],
    'original': hr_missing['heart_rate'],
    'forward_fill': hr_missing['heart_rate'].fillna(method='ffill'),
    'backward_fill': hr_missing['heart_rate'].fillna(method='bfill'),
    'linear_interpolation': hr_missing['heart_rate'].interpolate(method='linear'),
    'spline_interpolation': hr_missing['heart_rate'].interpolate(method='spline', order=3)
})

# Plot different imputation methods
plt.figure(figsize=(12, 10))

# Original with missing values
plt.subplot(5, 1, 1)
plt.plot(hr_imputed['datetime'], hr_imputed['original'], 'b-')
plt.title('Original (with missing values)')
plt.ylabel('Heart Rate')

# Forward fill
plt.subplot(5, 1, 2)
plt.plot(hr_imputed['datetime'], hr_imputed['forward_fill'], 'g-')
plt.title('Forward Fill')
plt.ylabel('Heart Rate')

# Backward fill
plt.subplot(5, 1, 3)
plt.plot(hr_imputed['datetime'], hr_imputed['backward_fill'], 'r-')
plt.title('Backward Fill')
plt.ylabel('Heart Rate')

# Linear interpolation
plt.subplot(5, 1, 4)
plt.plot(hr_imputed['datetime'], hr_imputed['linear_interpolation'], 'c-')
plt.title('Linear Interpolation')
plt.ylabel('Heart Rate')

# Spline interpolation
plt.subplot(5, 1, 5)
plt.plot(hr_imputed['datetime'], hr_imputed['spline_interpolation'], 'm-')
plt.title('Spline Interpolation')
plt.ylabel('Heart Rate')
plt.xlabel('Time')

plt.tight_layout()
plt.show()

Let's zoom in on a section to see the differences more clearly.

In [ ]:
# Zoom in on a section with missing values
start_idx = 100
end_idx = 200

plt.figure(figsize=(12, 6))
plt.plot(hr_imputed['datetime'][start_idx:end_idx], hr_imputed['original'][start_idx:end_idx], 'bo', alpha=0.5, label='Original')
plt.plot(hr_imputed['datetime'][start_idx:end_idx], hr_imputed['forward_fill'][start_idx:end_idx], 'g-', label='Forward Fill')
plt.plot(hr_imputed['datetime'][start_idx:end_idx], hr_imputed['linear_interpolation'][start_idx:end_idx], 'r-', label='Linear Interpolation')
plt.plot(hr_imputed['datetime'][start_idx:end_idx], hr_imputed['spline_interpolation'][start_idx:end_idx], 'm-', label='Spline Interpolation')

plt.title('Comparison of Imputation Methods (Zoomed)')
plt.xlabel('Time')
plt.ylabel('Heart Rate (bpm)')
plt.legend()
plt.grid(True)
plt.show()

## 5. Heart Rate Variability Analysis

Heart Rate Variability (HRV) is an important metric in meditation studies. Let's calculate and visualize it.

In [ ]:
# Calculate heart rate variability (standard deviation over a rolling window)
window_size = 60  # 60-second window
hr_data['hrv'] = hr_data['heart_rate'].rolling(window=window_size).std()

# Plot heart rate and HRV
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Heart rate
ax1.plot(hr_data['datetime'], hr_data['heart_rate'], 'b-', alpha=0.3)
ax1.plot(hr_data['datetime'], hr_data['hr_rolling'], 'r-', linewidth=2)
ax1.set_title('Heart Rate')
ax1.set_ylabel('Heart Rate (bpm)')
ax1.grid(True)

# Heart rate variability
ax2.plot(hr_data['datetime'], hr_data['hrv'], 'g-')
ax2.set_title('Heart Rate Variability (Rolling Standard Deviation)')
ax2.set_xlabel('Time')
ax2.set_ylabel('HRV (bpm)')
ax2.grid(True)

plt.tight_layout()
plt.show()

## 6. Summary and Discussion

In this demo, we've explored heart rate data during meditation. Here's what we've learned:

1. **Data Loading and Exploration**: We loaded real heart rate data from the PhysioNet Meditation dataset and explored its basic properties.

2. **Visualization**: We visualized heart rate patterns during meditation, comparing different subjects and meditation techniques.

3. **Statistical Analysis**: We performed stationarity tests and autocorrelation analysis to understand the time series properties.

4. **Handling Missing Values**: We demonstrated different methods for handling missing values in time series data, including forward fill, backward fill, and interpolation.

5. **Heart Rate Variability**: We calculated and visualized heart rate variability, an important metric in meditation studies.

### Key Insights:

- Heart rate patterns during meditation show interesting dynamics that can be analyzed with time series techniques.
- Different meditation techniques may lead to different heart rate patterns.
- Missing values in physiological data can be handled with various imputation methods, with interpolation often providing the most natural results.
- Heart rate variability provides additional insights beyond just the heart rate itself.

### Next Steps:

- Apply these techniques to your own physiological time series data.
- Explore more advanced time series analysis methods, such as frequency domain analysis.
- Consider how these patterns might relate to health outcomes or meditation effectiveness.